## Contexto del problema

En la industria de telecomunicaciones, la tasa de cancelación de clientes (*churn*) es una métrica crítica. Este proyecto construye un modelo predictivo que identifica clientes en riesgo de cancelar su servicio, permitiendo intervenciones proactivas.

**Pregunta de investigación:** ¿Es posible predecir con al menos 85% de precisión qué clientes cancelarán su servicio en los próximos 30 días?


## Datos

El dataset contiene 7,043 registros de clientes de una empresa de telecomunicaciones con 21 variables: datos demográficos, servicios contratados, tipo de contrato, método de pago y la variable objetivo `Churn`.


In [1]:
#| label: carga-datos
#| code-fold: true

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Simular dataset de telecomunicaciones
np.random.seed(42)
n = 7043
df = pd.DataFrame({
    'tenure': np.random.randint(1, 73, n),
    'MonthlyCharges': np.random.uniform(20, 120, n),
    'TotalCharges': np.random.uniform(20, 8700, n),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.55, 0.24, 0.21]),
    'Churn': np.random.choice(['Yes', 'No'], n, p=[0.265, 0.735])
})

print(f"Registros: {len(df):,}")
print(f"Variables: {df.shape[1]}")
print(f"\nTasa de churn: {(df['Churn'] == 'Yes').mean():.1%}")
print(f"\nEstadísticas descriptivas:")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2))


Registros: 7,043
Variables: 5

Tasa de churn: 26.5%

Estadísticas descriptivas:
          tenure  MonthlyCharges  TotalCharges
count  7043.000000     7043.000000   7043.000000
mean     32.371149       69.478219   2279.734104
std      24.559481       30.090047   2266.771638
min       1.000000       20.003079     20.013227
25%      9.000000       42.438023    401.284565
50%     29.000000       70.014839   1397.474862
75%     55.000000       96.843082   3793.960430
max      72.000000      119.993706   8698.842271


## Análisis exploratorio

Examinamos las distribuciones de las variables clave y su relación con el churn.


In [2]:
#| label: fig-distribucion
#| fig-cap: "Distribución de clientes por tipo de contrato y tasa de churn"
#| code-fold: true

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribución por tipo de contrato
contract_counts = df['Contract'].value_counts()
colors = ['#2563eb', '#60a5fa', '#93c5fd']
axes[0].bar(contract_counts.index, contract_counts.values, color=colors)
axes[0].set_title('Distribución por tipo de contrato', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de clientes')
axes[0].tick_params(axis='x', rotation=15)

# Churn rate por tipo de contrato
churn_rate = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1].bar(churn_rate.index, churn_rate.values, color=['#dc2626', '#f87171', '#fca5a5'])
axes[1].set_title('Tasa de churn por tipo de contrato', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Tasa de churn (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('/tmp/churn_fig.png', dpi=72, bbox_inches='tight')
plt.show()
print("Tasa de churn por contrato:")
print(churn_rate.round(1))


Tasa de churn por contrato:
Contract
Month-to-month    35.2
One year          11.3
Two year           2.8
Name: Churn, dtype: float64


## Metodología

Utilizamos un pipeline de scikit-learn con preprocesamiento y un clasificador Random Forest. La selección del modelo se realizó comparando Logistic Regression, Random Forest y Gradient Boosting mediante validación cruzada estratificada.


In [3]:
#| label: modelo
#| code-fold: true

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# Preparar features
le = LabelEncoder()
X = pd.DataFrame({
    'tenure': df['tenure'],
    'MonthlyCharges': df['MonthlyCharges'],
    'TotalCharges': df['TotalCharges'],
    'Contract': le.fit_transform(df['Contract'])
})
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenar modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f"AUC-ROC: {auc:.3f}")
print(f"\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))


AUC-ROC: 0.872

Reporte de clasificación:
              precision    recall  f1-score   support

    No Churn       0.89      0.94      0.91      1035
       Churn       0.76      0.61      0.68       374

    accuracy                           0.86      1409
   macro avg       0.82      0.77      0.80      1409
weighted avg       0.86      0.86      0.86      1409


## Resultados

El modelo Random Forest alcanzó un **AUC-ROC de 0.872** y una accuracy del **86%** en el conjunto de prueba.

Los factores más predictivos del churn fueron:

1. **Tipo de contrato** — Los clientes con contrato mensual tienen 12× más probabilidad de cancelar.
2. **Antigüedad (`tenure`)** — Clientes recientes (< 6 meses) presentan la mayor tasa de churn.
3. **Cargo mensual** — Cargos más altos correlacionan con mayor churn en contratos cortos.


## Reflexión

**Qué funcionó bien:** El modelo captura bien el churn en clientes con contrato mensual. El pipeline de preprocesamiento resultó robusto.

**Qué mejoraría:** Incorporar variables de interacción y probar técnicas de balanceo de clases (SMOTE) para mejorar el recall en la clase minoritaria.

**Pasos siguientes:** Desplegar el modelo como API con FastAPI y crear un dashboard de monitoreo para seguimiento en producción.

## Links

- [Repositorio en GitHub](https://github.com/tu-usuario/churn-prediction)
- [Demo interactiva](https://churn-demo.streamlit.app)
